# Firecrawl

## Web search

In [ ]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

FIRECRAWL_API = os.environ["FIRECRAWL_API_KEY"]

url = "https://api.firecrawl.dev/v2/search"

payload = {
    "query": "what are the latest news about Jensen Huang?",
    "sources": ["web"],
    "categories": [],
    "limit": 3,
    "scrapeOptions": {
        "onlyMainContent": True,
        "maxAge": 172800000,
        "parsers": [
         "pdf"],
        "formats": []
  }
}

headers = {
    "Authorization": f"Bearer {FIRECRAWL_API}",
    "Content-Type": "application/json"
}

response = requests.post(url, json=payload, headers=headers)

print(response.json())

In [2]:
response.json().keys()

dict_keys(['success', 'data', 'creditsUsed', 'id'])

In [3]:
response.json()["creditsUsed"]

2

In [4]:
response.json()["data"].keys()

dict_keys(['web'])

In [5]:
search_results = response.json()["data"]["web"]
len(search_results)

3

In [6]:
search_results[0].keys

<function dict.keys()>

In [7]:
search_results[0]["url"]

'https://www.cbsnews.com/news/jensen-huang-nvidia-ceo-return-statement/'

In [8]:
search_results[0]["title"]

'Jensen Huang breaks silence after resigning from Nvidia, says he " ...'

In [9]:
search_results[0]["position"]

1

In [ ]:
print(search_results[0]["description"])

In [11]:
import json

with open("jensen_huang.json", "w", encoding="utf-8") as f:
    json.dump(search_results, f, ensure_ascii=False, indent=4)

In [ ]:
import json
with open("jensen_huang.json", encoding="utf-8") as f:
    data = json.load(f)

data

## Webscraping

- scrape a web page

In [ ]:
from firecrawl import Firecrawl

app = Firecrawl()

data = app.scrape(
    "https://pydantic.dev/docs/ai/evals/evals/",
    only_main_content=True,
    max_age=172800000,
    parsers=["pdf"],
    formats=["markdown"],
)

print(data)

In [14]:
data.__dict__.keys()

dict_keys(['agent_hints', 'markdown', 'html', 'raw_html', 'json', 'summary', 'metadata', 'links', 'images', 'screenshot', 'audio', 'video', 'actions', 'answer', 'highlights', 'warning', 'change_tracking', 'branding', 'product', 'menu', 'pages', 'blocks', 'tools'])

In [ ]:
metadata = data.metadata
metadata.__dict__.keys()

In [18]:
metadata.title

'Pydantic Evals | Pydantic Docs'

In [21]:
with open(f"{metadata.title.replace("|", "")}.md", "w", encoding=("utf-8")) as f:
    f.write(data.markdown)

## Crawling

In [22]:
from firecrawl import Firecrawl
from firecrawl.v2.types import ScrapeOptions

app = Firecrawl()

scrape_opts = ScrapeOptions(
    only_main_content=True,
    max_age=172800000,
    parsers=["pdf"]
)

crawl_result = app.crawl(
    "https://www.myh.se/validering-och-seqf",
    sitemap="include",
    crawl_entire_domain=False,
    limit=10,
    scrape_options=scrape_opts
)

In [23]:
crawl_result.__dict__.keys()

dict_keys(['id', 'status', 'total', 'completed', 'credits_used', 'expires_at', 'next', 'warning', 'data'])

In [25]:
crawl_result.credits_used

10

In [ ]:
from pathlib import Path

Path("myh").mkdir(exist_ok=True)

for document in crawl_result.data:
    metadata = document.metadata
    with open(f"myh/{metadata.title.replace('|', '')}.md", "w", encoding="utf-8") as f:
        f.write(document.markdown)

[Document(agent_hints=None, markdown='[Till innehållet](https://www.myh.se/validering-och-seqf#main-content)\n\n# Validering och SeQF\n\nPå de här sidorna hittar du information om validering och om SeQF, Sveriges referensram för kvalifikationer för livslångt lärande. Här finns information dels för dig som vill valideras, dels för dig som kommer i kontakt med olika slags validering och SeQF i ditt arbete.\n\nMyndigheten för yrkeshögskolan ansvarar för frågor om olika slags valideringar och för SeQF. I sidmenyn kan du som vill veta mer hitta vägar vidare till mer information.\n\nMyndigheten för yrkeshögskolan anordnar konferenser och seminarier inom ramen för validering och SeQF. Dem kan du läsa mer om här på våra sidor. Du kan också läsa om våra projekt som pågår inom ämnesområdena.\n\n## Validering\n\nValidering är ett verktyg för att synliggöra och erkänna individers faktiska kompetens – oavsett var, hur eller när den kompetensen utvecklats.\n\nValidering kan vara lämpligt för den som